In [60]:
import bz2
import json
import os
import glob
import pandas as pd

In [61]:
data_folder = "six_nations_data"

files = glob.glob(os.path.join(data_folder, "*.bz2"))

print("Number of files:", len(files))

files[:10]

Number of files: 15


['six_nations_data/1.254672456.bz2',
 'six_nations_data/1.254269279.bz2',
 'six_nations_data/1.254110200.bz2',
 'six_nations_data/1.253655399.bz2',
 'six_nations_data/1.254552610.bz2',
 'six_nations_data/1.255310340.bz2',
 'six_nations_data/1.253772842.bz2',
 'six_nations_data/1.255461895.bz2',
 'six_nations_data/1.254890715.bz2',
 'six_nations_data/1.253638185.bz2']

In [62]:
market_information = []

for file_path in files:

    market_info = get_market_info(file_path)

    if market_info is not None:
        market_information.append(market_info)

markets_df = pd.DataFrame(market_information)

In [63]:
def get_market_info(file_path):

    with bz2.open(file_path, "rt") as f:

        for line in f:

            data = json.loads(line)

            for market_change in data.get("mc", []):

                market_definition = market_change.get(
                    "marketDefinition"
                )

                if market_definition:

                    runners = market_definition.get("runners", [])

                    return {
                        "File": os.path.basename(file_path),
                        "Market ID": market_change.get("id"),
                        "Event": market_definition.get("eventName"),
                        "Market": market_definition.get("name"),
                        "Market Type": market_definition.get("marketType"),
                        "Kick-off": market_definition.get("marketTime"),
                        "Runners": [
                            runner.get("name")
                            for runner in runners
                        ]
                    }

    return None

In [64]:
# Get the scheduled kick-off time
kick_off = pd.to_datetime(
    markets_df["Kick-off"],
    utc=True
)

# Calculate one hour before kick-off
target_time = kick_off - pd.Timedelta(hours=1)

In [65]:
def get_ltp_at_time(file_path, target_time):

    runner_names = {}
    latest_prices = {}

    target_ms = int(target_time.timestamp() * 1000)

    with bz2.open(file_path, "rt") as f:

        for line in f:

            data = json.loads(line)

            timestamp = data.get("pt")

            if timestamp is None or timestamp > target_ms:
                continue

            for mc in data.get("mc", []):

                definition = mc.get("marketDefinition", {})

                for runner in definition.get("runners", []):
                    runner_names[runner["id"]] = runner["name"]

                for runner_change in mc.get("rc", []):

                    selection_id = runner_change["id"]

                    if "ltp" in runner_change:
                        latest_prices[selection_id] = runner_change["ltp"]

    return [
        {
            "Selection": runner_names.get(selection_id),
            "Last Traded Price": price
        }
        for selection_id, price in latest_prices.items()
    ]

In [66]:
ireland_senior_matches = markets_df[
    markets_df["Event"].str.contains(
        "Ireland",
        case=False,
        na=False
    )
    &
    ~markets_df["Event"].str.contains(
        "U20|Women|XV|Azerbaijan|Luxembourg",
        case=False,
        na=False
    )
]

ireland_senior_matches[
    ["Event", "Market ID", "Kick-off", "Runners"]
]

,Event,Market ID,Kick-off,Runners
2,England v Ireland,1.254110200,2026-02-21T14:10:00.000Z,"[England, Ireland, Draw]"
6,Ireland v Italy,1.253772842,2026-02-14T14:10:00.000Z,"[Ireland, Italy, Draw]"
10,Ireland v Scotland,1.254952251,2026-03-14T14:10:00.000Z,"[Ireland, Scotland, Draw]"
12,France v Ireland,1.251244067,2026-02-05T20:10:00.000Z,"[France, Ireland, Draw]"
14,Ireland v Wales,1.254392836,2026-03-06T20:10:00.000Z,"[Ireland, Wales, Draw]"


In [67]:
ireland_senior_matches = ireland_senior_matches.sort_values(
    "Kick-off"
).reset_index(drop=True)

ireland_senior_matches[
    ["Event", "Market ID", "Kick-off"]
]

def get_ltp_at_time(file_path, target_time):

    runner_names = {}
    latest_prices = {}

    target_ms = int(target_time.timestamp() * 1000)

    with bz2.open(file_path, "rt") as f:

        for line in f:

            data = json.loads(line)

            timestamp = data.get("pt")

            if timestamp is None or timestamp > target_ms:
                continue

            for mc in data.get("mc", []):

                definition = mc.get("marketDefinition", {})
    
                for runner in definition.get("runners", []):
                    runner_names[runner["id"]] = runner["name"]

                for runner_change in mc.get("rc", []):

                    selection_id = runner_change["id"]

                    if "ltp" in runner_change:
                        latest_prices[selection_id] = runner_change["ltp"]

    return [
        {
            "Selection": runner_names.get(selection_id),
            "Last Traded Price": price
        }
        for selection_id, price in latest_prices.items()
    ]

In [68]:
results = []

for _, market in ireland_senior_matches.iterrows():

    kick_off = pd.to_datetime(market["Kick-off"], utc=True)

    target_time = kick_off - pd.Timedelta(hours=1)

    file_path = os.path.join(
        data_folder,
        market["File"]
    )

    prices = get_ltp_at_time(file_path, target_time)

    for price in prices:

        results.append({
            "Match": market["Event"],
            "Kick-off": kick_off,
            "Target Time": target_time,
            "Selection": price["Selection"],
            "Last Traded Price": price["Last Traded Price"]
        })

odds_df = pd.DataFrame(results)

odds_df

,Match,Kick-off,Target Time,Selection,Last Traded Price
0,France v Ireland,2026-02-05 20:10:00+00:00,2026-02-05 19:10:00+00:00,France,1.23
1,France v Ireland,2026-02-05 20:10:00+00:00,2026-02-05 19:10:00+00:00,Ireland,6.00
2,France v Ireland,2026-02-05 20:10:00+00:00,2026-02-05 19:10:00+00:00,Draw,48.00
3,Ireland v Italy,2026-02-14 14:10:00+00:00,2026-02-14 13:10:00+00:00,Ireland,1.11
4,Ireland v Italy,2026-02-14 14:10:00+00:00,2026-02-14 13:10:00+00:00,Italy,10.50
5,Ireland v Italy,2026-02-14 14:10:00+00:00,2026-02-14 13:10:00+00:00,Draw,80.00
6,England v Ireland,2026-02-21 14:10:00+00:00,2026-02-21 13:10:00+00:00,Ireland,4.20
7,England v Ireland,2026-02-21 14:10:00+00:00,2026-02-21 13:10:00+00:00,England,1.36
8,England v Ireland,2026-02-21 14:10:00+00:00,2026-02-21 13:10:00+00:00,Draw,40.00
9,Ireland v Wales,2026-03-06 20:10:00+00:00,2026-03-06 19:10:00+00:00,Wales,27.00


In [69]:
import bz2
import json
import os
import glob
import pandas as pd
from collections import Counter

data_folder = "six_nations_data"

# Minutes before scheduled kick-off
snapshot_minutes = [180, 120, 60, 30, 15, 5, 0]

files = glob.glob(os.path.join(data_folder, "*.bz2"))

print("Number of files:", len(files))

Number of files: 15


In [70]:
def get_market_info(file_path):

    with bz2.open(file_path, "rt") as f:

        for line in f:

            data = json.loads(line)

            for market_change in data.get("mc", []):

                market_definition = market_change.get("marketDefinition")

                if market_definition:

                    runners = market_definition.get("runners", [])

                    return {
                        "File": os.path.basename(file_path),
                        "Market ID": market_change.get("id"),
                        "Event": market_definition.get("eventName"),
                        "Market": market_definition.get("name"),
                        "Market Type": market_definition.get("marketType"),
                        "Kick-off": market_definition.get("marketTime"),
                        "Runners": [
                            runner.get("name")
                            for runner in runners
                        ]
                    }

    return None


market_information = []

for file_path in files:

    market_info = get_market_info(file_path)

    if market_info is not None:
        market_information.append(market_info)

markets_df = pd.DataFrame(market_information)

print("Markets found:", len(markets_df))

Markets found: 15


In [71]:
ireland_senior_matches = markets_df[
    markets_df["Event"].str.contains(
        "Ireland",
        case=False,
        na=False
    )
    &
    ~markets_df["Event"].str.contains(
        "U20|Women|XV|Azerbaijan|Luxembourg",
        case=False,
        na=False
    )
    &
    (markets_df["Market Type"] == "MATCH_ODDS")
].copy()

ireland_senior_matches["Kick-off"] = pd.to_datetime(
    ireland_senior_matches["Kick-off"],
    utc=True
)

ireland_senior_matches = (
    ireland_senior_matches
    .sort_values("Kick-off")
    .reset_index(drop=True)
)

ireland_senior_matches[
    ["Event", "Market ID", "Market Type", "Kick-off", "Runners"]
]

,Event,Market ID,Market Type,Kick-off,Runners
0,France v Ireland,1.251244067,MATCH_ODDS,2026-02-05 20:10:00+00:00,"[France, Ireland, Draw]"
1,Ireland v Italy,1.253772842,MATCH_ODDS,2026-02-14 14:10:00+00:00,"[Ireland, Italy, Draw]"
2,England v Ireland,1.254110200,MATCH_ODDS,2026-02-21 14:10:00+00:00,"[England, Ireland, Draw]"
3,Ireland v Wales,1.254392836,MATCH_ODDS,2026-03-06 20:10:00+00:00,"[Ireland, Wales, Draw]"
4,Ireland v Scotland,1.254952251,MATCH_ODDS,2026-03-14 14:10:00+00:00,"[Ireland, Scotland, Draw]"


In [72]:
def get_ltp_at_multiple_times(file_path, kick_off, snapshot_minutes):

    # Calculate target timestamps
    targets = [
        {
            "minutes_before": minutes,
            "target_time": kick_off - pd.Timedelta(minutes=minutes),
            "target_ms": int(
                (kick_off - pd.Timedelta(minutes=minutes)).timestamp() * 1000
            )
        }
        for minutes in snapshot_minutes
    ]

    targets = sorted(targets, key=lambda x: x["target_ms"])

    target_index = 0

    runner_names = {}
    latest_prices = {}
    last_ltp_timestamps = {}

    results = []

    # Audit the fields actually present in the historical file
    runner_change_keys = Counter()

    def save_snapshot(target):

        for selection_id, selection_name in runner_names.items():

            last_price = latest_prices.get(selection_id)
            last_price_ms = last_ltp_timestamps.get(selection_id)

            if last_price_ms is not None:

                last_traded_at = pd.to_datetime(
                    last_price_ms,
                    unit="ms",
                    utc=True
                )

                age_minutes = (
                    target["target_ms"] - last_price_ms
                ) / 60000

            else:
                last_traded_at = pd.NaT
                age_minutes = None

            results.append({
                "Selection": selection_name,
                "Selection ID": selection_id,
                "Minutes Before Kick-off": target["minutes_before"],
                "Snapshot Time": target["target_time"],
                "Last Traded Price": last_price,
                "Last Traded At": last_traded_at,
                "Minutes Since Last Trade": age_minutes
            })

    with bz2.open(file_path, "rt") as f:

        current_timestamp = None

        for line in f:

            data = json.loads(line)

            timestamp = data.get("pt")

            if timestamp is None:
                continue

            timestamp = int(timestamp)

            # Before processing a newer timestamp, save any
            # target times that have already passed.
            if current_timestamp is None or timestamp != current_timestamp:

                while (
                    target_index < len(targets)
                    and targets[target_index]["target_ms"] < timestamp
                ):

                    save_snapshot(targets[target_index])
                    target_index += 1

                current_timestamp = timestamp

            for market_change in data.get("mc", []):

                market_definition = market_change.get("marketDefinition")

                if market_definition:

                    for runner in market_definition.get("runners", []):

                        selection_id = runner.get("id")

                        if selection_id is not None:
                            runner_names[selection_id] = runner.get("name")

                for runner_change in market_change.get("rc", []):

                    runner_change_keys.update(runner_change.keys())

                    selection_id = runner_change.get("id")

                    if selection_id is None:
                        continue

                    if "ltp" in runner_change:

                        latest_prices[selection_id] = runner_change["ltp"]

                        last_ltp_timestamps[selection_id] = timestamp

    # Save any remaining target times
    while target_index < len(targets):

        save_snapshot(targets[target_index])
        target_index += 1

    return results, runner_change_keys

In [73]:
all_results = []
all_field_counts = Counter()

for _, market in ireland_senior_matches.iterrows():

    kick_off = market["Kick-off"]

    file_path = os.path.join(
        data_folder,
        market["File"]
    )

    prices, field_counts = get_ltp_at_multiple_times(
        file_path,
        kick_off,
        snapshot_minutes
    )

    all_field_counts.update(field_counts)

    for price in prices:

        all_results.append({
            "Match": market["Event"],
            "Market ID": market["Market ID"],
            "Kick-off": kick_off,
            **price
        })


price_snapshots_df = pd.DataFrame(all_results)

price_snapshots_df.head(20)

,Match,Market ID,Kick-off,Selection,Selection ID,Minutes Before Kick-off,Snapshot Time,Last Traded Price,Last Traded At,Minutes Since Last Trade
0,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,France,13098,180,2026-02-05 17:10:00+00:00,1.22,2026-02-05 17:06:15.106+00:00,3.748233
1,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,Ireland,351,180,2026-02-05 17:10:00+00:00,6.20,2026-02-05 17:05:06.236+00:00,4.896067
2,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,Draw,246289,180,2026-02-05 17:10:00+00:00,46.00,2026-02-05 17:09:16.034+00:00,0.732767
3,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,France,13098,120,2026-02-05 18:10:00+00:00,1.21,2026-02-05 18:04:16.544+00:00,5.724267
4,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,Ireland,351,120,2026-02-05 18:10:00+00:00,6.20,2026-02-05 18:07:16.375+00:00,2.727083
5,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,Draw,246289,120,2026-02-05 18:10:00+00:00,46.00,2026-02-05 18:09:16.413+00:00,0.726450
6,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,France,13098,60,2026-02-05 19:10:00+00:00,1.23,2026-02-05 19:06:16.470+00:00,3.725500
7,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,Ireland,351,60,2026-02-05 19:10:00+00:00,6.00,2026-02-05 18:56:16.415+00:00,13.726417
8,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,Draw,246289,60,2026-02-05 19:10:00+00:00,48.00,2026-02-05 19:06:16.470+00:00,3.725500
9,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,France,13098,30,2026-02-05 19:40:00+00:00,1.23,2026-02-05 19:37:16.820+00:00,2.719667


In [74]:
price_snapshots_df = price_snapshots_df.sort_values(
    ["Kick-off", "Selection", "Minutes Before Kick-off"],
    ascending=[True, True, False]
).reset_index(drop=True)

price_snapshots_df[
    [
        "Match",
        "Selection",
        "Minutes Before Kick-off",
        "Last Traded Price",
        "Last Traded At",
        "Minutes Since Last Trade"
    ]
]


,Match,Selection,Minutes Before Kick-off,Last Traded Price,Last Traded At,Minutes Since Last Trade
0,France v Ireland,Draw,180,46.00,2026-02-05 17:09:16.034+00:00,0.732767
1,France v Ireland,Draw,120,46.00,2026-02-05 18:09:16.413+00:00,0.726450
2,France v Ireland,Draw,60,48.00,2026-02-05 19:06:16.470+00:00,3.725500
3,France v Ireland,Draw,30,50.00,2026-02-05 19:35:16.775+00:00,4.720417
4,France v Ireland,Draw,15,48.00,2026-02-05 19:54:16.837+00:00,0.719383
...,...,...,...,...,...,...
100,Ireland v Scotland,Scotland,60,3.35,2026-03-14 13:09:41.008+00:00,0.316533
101,Ireland v Scotland,Scotland,30,3.40,2026-03-14 13:38:40.144+00:00,1.330933
102,Ireland v Scotland,Scotland,15,3.35,2026-03-14 13:40:41.385+00:00,14.310250
103,Ireland v Scotland,Scotland,5,3.35,2026-03-14 14:02:42.441+00:00,2.292650


In [75]:
price_movement_df = price_snapshots_df.pivot_table(
    index=["Match", "Selection"],
    columns="Minutes Before Kick-off",
    values="Last Traded Price",
    aggfunc="last"
)

price_movement_df = price_movement_df.rename(
    columns=lambda x: f"T-{x}"
)

price_movement_df

Minutes Before Kick-off          T-0     T-5    T-15    T-30    T-60  T-120  \
Match              Selection                                                  
England v Ireland  Draw        40.00   38.00   38.00   38.00   40.00  36.00   
                   England      1.35    1.33    1.33    1.34    1.36   1.34   
                   Ireland      4.30    4.40    4.50    4.50    4.20   4.40   
France v Ireland   Draw        55.00   48.00   48.00   50.00   48.00  46.00   
                   France       1.22    1.23    1.22    1.23    1.23   1.21   
                   Ireland      6.00    5.90    6.00    6.20    6.00   6.20   
Ireland v Italy    Draw        80.00   70.00   65.00   75.00   80.00  60.00   
                   Ireland      1.11    1.11    1.12    1.12    1.11   1.12   
                   Italy       11.00   10.50   10.50   10.50   10.50  11.00   
Ireland v Scotland Draw        34.00   34.00   32.00   32.00   29.00  29.00   
                   Ireland      1.47    1.48    1.48    1.48    1.48   1.48   
                   Scotland     3.40    3.35    3.35    3.40    3.35   3.35   
Ireland v Wales    Draw       130.00  130.00  150.00  100.00  100.00  95.00   
                   Ireland      1.05    1.05    1.04    1.04    1.04   1.05   
                   Wales       25.00   27.00   28.00   26.00   27.00  24.00   

Minutes Before Kick-off       T-180  
Match              Selection         
England v Ireland  Draw       36.00  
                   England     1.32  
                   Ireland     4.50  
France v Ireland   Draw       46.00  
                   France      1.22  
                   Ireland     6.20  
Ireland v Italy    Draw       50.00  
                   Ireland     1.13  
                   Italy      10.00  
Ireland v Scotland Draw       28.00  
                   Ireland     1.49  
                   Scotland    3.40  
Ireland v Wales    Draw       90.00  
                   Ireland     1.05  
                   Wales      22.00

In [76]:
if "T-60" in price_movement_df.columns and "T-5" in price_movement_df.columns:

    price_movement_df["Price Change % (T-60 to T-5)"] = (
        (
            price_movement_df["T-5"]
            - price_movement_df["T-60"]
        )
        / price_movement_df["T-60"]
    ) * 100

    price_movement_df["Movement"] = price_movement_df[
        "Price Change % (T-60 to T-5)"
    ].apply(
        lambda x:
            "Shortened" if x < 0
            else "Drifted" if x > 0
            else "Unchanged" if pd.notna(x)
            else "Insufficient data"
    )

price_movement_df

Minutes Before Kick-off          T-0     T-5    T-15    T-30    T-60  T-120  \
Match              Selection                                                  
England v Ireland  Draw        40.00   38.00   38.00   38.00   40.00  36.00   
                   England      1.35    1.33    1.33    1.34    1.36   1.34   
                   Ireland      4.30    4.40    4.50    4.50    4.20   4.40   
France v Ireland   Draw        55.00   48.00   48.00   50.00   48.00  46.00   
                   France       1.22    1.23    1.22    1.23    1.23   1.21   
                   Ireland      6.00    5.90    6.00    6.20    6.00   6.20   
Ireland v Italy    Draw        80.00   70.00   65.00   75.00   80.00  60.00   
                   Ireland      1.11    1.11    1.12    1.12    1.11   1.12   
                   Italy       11.00   10.50   10.50   10.50   10.50  11.00   
Ireland v Scotland Draw        34.00   34.00   32.00   32.00   29.00  29.00   
                   Ireland      1.47    1.48    1.48    1.48    1.48   1.48   
                   Scotland     3.40    3.35    3.35    3.40    3.35   3.35   
Ireland v Wales    Draw       130.00  130.00  150.00  100.00  100.00  95.00   
                   Ireland      1.05    1.05    1.04    1.04    1.04   1.05   
                   Wales       25.00   27.00   28.00   26.00   27.00  24.00   

Minutes Before Kick-off       T-180  Price Change % (T-60 to T-5)   Movement  
Match              Selection                                                  
England v Ireland  Draw       36.00                     -5.000000  Shortened  
                   England     1.32                     -2.205882  Shortened  
                   Ireland     4.50                      4.761905    Drifted  
France v Ireland   Draw       46.00                      0.000000  Unchanged  
                   France      1.22                      0.000000  Unchanged  
                   Ireland     6.20                     -1.666667  Shortened  
Ireland v Italy    Draw       50.00                    -12.500000  Shortened  
                   Ireland     1.13                      0.000000  Unchanged  
                   Italy      10.00                      0.000000  Unchanged  
Ireland v Scotland Draw       28.00                     17.241379    Drifted  
                   Ireland     1.49                      0.000000  Unchanged  
                   Scotland    3.40                      0.000000  Unchanged  
Ireland v Wales    Draw       90.00                     30.000000    Drifted  
                   Ireland     1.05                      0.961538    Drifted  
                   Wales      22.00                      0.000000  Unchanged

In [77]:
interesting_fields = [
    "ltp",
    "tv",
    "atb",
    "atl",
    "batb",
    "batl",
    "bdatb",
    "bdatl",
    "trd"
]

field_audit = pd.DataFrame([
    {
        "Field": field,
        "Number of occurrences": all_field_counts.get(field, 0)
    }
    for field in interesting_fields
])

field_audit

,Field,Number of occurrences
0,ltp,4295
1,tv,0
2,atb,0
3,atl,0
4,batb,0
5,batl,0
6,bdatb,0
7,bdatl,0
8,trd,0


In [78]:
price_snapshots_df.to_csv(
    "ireland_six_nations_price_snapshots.csv",
    index=False
)

price_movement_df.to_csv(
    "ireland_six_nations_price_movement.csv"
)

field_audit.to_csv(
    "ireland_six_nations_field_audit.csv",
    index=False
)

print("Files saved successfully.")

Files saved successfully.


In [79]:
ENTRY_BACK_STAKE = 10.00
MAX_LAY_LIABILITY = 10.00

COMMISSION_RATE = 0.05

ENTRY_MINUTES = 60
EXIT_MINUTES = 5

MAX_PRICE_AGE_MINUTES = None

In [80]:
trade_data = price_snapshots_df[
    price_snapshots_df["Minutes Before Kick-off"].isin(
        [ENTRY_MINUTES, EXIT_MINUTES]
    )
].copy()

trade_prices = trade_data.pivot_table(
    index=[
        "Match",
        "Market ID",
        "Kick-off",
        "Selection",
        "Selection ID"
    ],
    columns="Minutes Before Kick-off",
    values="Last Traded Price",
    aggfunc="last"
)

trade_ages = trade_data.pivot_table(
    index=[
        "Match",
        "Market ID",
        "Kick-off",
        "Selection",
        "Selection ID"
    ],
    columns="Minutes Before Kick-off",
    values="Minutes Since Last Trade",
    aggfunc="last"
)

trade_prices = trade_prices.rename(
    columns={
        ENTRY_MINUTES: "Entry Odds",
        EXIT_MINUTES: "Exit Odds"
    }
)

trade_ages = trade_ages.rename(
    columns={
        ENTRY_MINUTES: "Entry Price Age (mins)",
        EXIT_MINUTES: "Exit Price Age (mins)"
    }
)

trade_data = trade_prices.join(trade_ages).reset_index()

trade_data.head()

Minutes Before Kick-off,Match,Market ID,Kick-off,Selection,Selection ID,Exit Odds,Entry Odds,Exit Price Age (mins),Entry Price Age (mins)
0,England v Ireland,1.254110200,2026-02-21 14:10:00+00:00,Draw,246289,38.00,40.00,11.258567,0.307533
1,England v Ireland,1.254110200,2026-02-21 14:10:00+00:00,England,15593,1.33,1.36,0.246383,0.307533
2,England v Ireland,1.254110200,2026-02-21 14:10:00+00:00,Ireland,351,4.40,4.20,1.259067,0.307533
3,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,Draw,246289,48.00,48.00,1.727833,3.725500
4,France v Ireland,1.251244067,2026-02-05 20:10:00+00:00,France,13098,1.23,1.23,0.726650,3.725500


In [81]:
trade_data["Valid Prices"] = (
    trade_data["Entry Odds"].notna()
    & trade_data["Exit Odds"].notna()
    & (trade_data["Entry Odds"] > 1)
    & (trade_data["Exit Odds"] > 1)
)

trade_data["Entry Price Fresh"] = (
    trade_data["Entry Price Age (mins)"].notna()
    & (trade_data["Entry Price Age (mins)"] >= 0)
)

trade_data["Exit Price Fresh"] = (
    trade_data["Exit Price Age (mins)"].notna()
    & (trade_data["Exit Price Age (mins)"] >= 0)
)

if MAX_PRICE_AGE_MINUTES is not None:

    trade_data["Entry Price Fresh"] &= (
        trade_data["Entry Price Age (mins)"]
        <= MAX_PRICE_AGE_MINUTES
    )

    trade_data["Exit Price Fresh"] &= (
        trade_data["Exit Price Age (mins)"]
        <= MAX_PRICE_AGE_MINUTES
    )

trade_data["Usable For Test"] = (
    trade_data["Valid Prices"]
    & trade_data["Entry Price Fresh"]
    & trade_data["Exit Price Fresh"]
)

trade_data[
    [
        "Match",
        "Selection",
        "Entry Odds",
        "Exit Odds",
        "Entry Price Age (mins)",
        "Exit Price Age (mins)",
        "Usable For Test"
    ]
]

Minutes Before Kick-off,Match,Selection,Entry Odds,Exit Odds,Entry Price Age (mins),Exit Price Age (mins),Usable For Test
0,England v Ireland,Draw,40.00,38.00,0.307533,11.258567,True
1,England v Ireland,England,1.36,1.33,0.307533,0.246383,True
2,England v Ireland,Ireland,4.20,4.40,0.307533,1.259067,True
3,France v Ireland,Draw,48.00,48.00,3.725500,1.727833,True
4,France v Ireland,France,1.23,1.23,3.725500,0.726650,True
5,France v Ireland,Ireland,6.00,5.90,13.726417,0.726650,True
6,Ireland v Italy,Draw,80.00,70.00,0.683267,6.676167,True
7,Ireland v Italy,Ireland,1.11,1.11,0.683267,9.687233,True
8,Ireland v Italy,Italy,10.50,10.50,5.681900,3.677333,True
9,Ireland v Scotland,Draw,29.00,34.00,5.327767,3.304167,True


In [82]:
def calculate_back_then_lay(entry_odds, exit_odds, back_stake):

    lay_stake = back_stake * entry_odds / exit_odds

    gross_profit = back_stake * (
        entry_odds / exit_odds - 1
    )

    commission = max(gross_profit, 0) * COMMISSION_RATE

    net_profit = gross_profit - commission

    return {
        "Entry Back Stake": back_stake,
        "Exit Lay Stake": lay_stake,
        "Exit Lay Liability": lay_stake * (exit_odds - 1),
        "Gross Green Profit": gross_profit,
        "Commission": commission,
        "Net Green Profit": net_profit
    }


def calculate_lay_then_back(entry_odds, exit_odds, max_liability):
    if entry_odds <= 1 or exit_odds <= 1:
        return None

    lay_stake = max_liability / (entry_odds - 1)

    back_stake = lay_stake * entry_odds / exit_odds

    gross_profit = lay_stake - back_stake

    commission = max(gross_profit, 0) * COMMISSION_RATE

    net_profit = gross_profit - commission

    return {
        "Entry Lay Stake": lay_stake,
        "Entry Lay Liability": max_liability,
        "Exit Back Stake": back_stake,
        "Gross Green Profit": gross_profit,
        "Commission": commission,
        "Net Green Profit": net_profit
    }

In [83]:
trade_results = []

for _, row in trade_data.iterrows():

    if not row["Usable For Test"]:
        continue

    entry_odds = row["Entry Odds"]
    exit_odds = row["Exit Odds"]

    back_lay = calculate_back_then_lay(
    entry_odds,
    exit_odds,
    ENTRY_BACK_STAKE)

    lay_back = calculate_lay_then_back(
    entry_odds,
    exit_odds,
    MAX_LAY_LIABILITY)

    common = {
        "Match": row["Match"],
        "Market ID": row["Market ID"],
        "Selection": row["Selection"],
        "Entry Time": f"T-{ENTRY_MINUTES}",
        "Exit Time": f"T-{EXIT_MINUTES}",
        "Entry Odds": entry_odds,
        "Exit Odds": exit_odds,
        "Price Change %": (
            (exit_odds - entry_odds) / entry_odds
        ) * 100,
        "Entry Price Age (mins)": row["Entry Price Age (mins)"],
        "Exit Price Age (mins)": row["Exit Price Age (mins)"]
    }

    trade_results.append({
        **common,
        "Strategy": "Back then Lay",
        **back_lay
    })

    trade_results.append({
        **common,
        "Strategy": "Lay then Back",
        **lay_back
    })


trades_df = pd.DataFrame(trade_results)

trades_df.head(20)

,Match,Market ID,Selection,Entry Time,Exit Time,Entry Odds,Exit Odds,Price Change %,Entry Price Age (mins),Exit Price Age (mins),Strategy,Entry Back Stake,Exit Lay Stake,Exit Lay Liability,Gross Green Profit,Commission,Net Green Profit,Entry Lay Stake,Entry Lay Liability,Exit Back Stake
0,England v Ireland,1.254110200,Draw,T-60,T-5,40.00,38.00,-5.000000,0.307533,11.258567,Back then Lay,10.0,10.526316,389.473684,0.526316,0.026316,0.500000,NaN,NaN,NaN
1,England v Ireland,1.254110200,Draw,T-60,T-5,40.00,38.00,-5.000000,0.307533,11.258567,Lay then Back,NaN,NaN,NaN,-0.013495,0.000000,-0.013495,0.256410,10.0,0.269906
2,England v Ireland,1.254110200,England,T-60,T-5,1.36,1.33,-2.205882,0.307533,0.246383,Back then Lay,10.0,10.225564,3.374436,0.225564,0.011278,0.214286,NaN,NaN,NaN
3,England v Ireland,1.254110200,England,T-60,T-5,1.36,1.33,-2.205882,0.307533,0.246383,Lay then Back,NaN,NaN,NaN,-0.626566,0.000000,-0.626566,27.777778,10.0,28.404344
4,England v Ireland,1.254110200,Ireland,T-60,T-5,4.20,4.40,4.761905,0.307533,1.259067,Back then Lay,10.0,9.545455,32.454545,-0.454545,0.000000,-0.454545,NaN,NaN,NaN
5,England v Ireland,1.254110200,Ireland,T-60,T-5,4.20,4.40,4.761905,0.307533,1.259067,Lay then Back,NaN,NaN,NaN,0.142045,0.007102,0.134943,3.125000,10.0,2.982955
6,France v Ireland,1.251244067,Draw,T-60,T-5,48.00,48.00,0.000000,3.725500,1.727833,Back then Lay,10.0,10.000000,470.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
7,France v Ireland,1.251244067,Draw,T-60,T-5,48.00,48.00,0.000000,3.725500,1.727833,Lay then Back,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.212766,10.0,0.212766
8,France v Ireland,1.251244067,France,T-60,T-5,1.23,1.23,0.000000,3.725500,0.726650,Back then Lay,10.0,10.000000,2.300000,0.000000,0.000000,0.000000,NaN,NaN,NaN
9,France v Ireland,1.251244067,France,T-60,T-5,1.23,1.23,0.000000,3.725500,0.726650,Lay then Back,NaN,NaN,NaN,0.000000,0.000000,0.000000,43.478261,10.0,43.478261


In [85]:
trades_df["Result"] = trades_df["Net Green Profit"].apply(
    lambda x:
        "Profit" if x > 0
        else "Loss" if x < 0
        else "Break-even"
)

trades_df[
    [
        "Match",
        "Selection",
        "Strategy",
        "Entry Odds",
        "Exit Odds",
        "Gross Green Profit",
        "Commission",
        "Net Green Profit",
        "Result"
    ]
].sort_values(
    ["Match", "Selection", "Strategy"]
)

,Match,Selection,Strategy,Entry Odds,Exit Odds,Gross Green Profit,Commission,Net Green Profit,Result
0,England v Ireland,Draw,Back then Lay,40.00,38.00,0.526316,0.026316,0.500000,Profit
1,England v Ireland,Draw,Lay then Back,40.00,38.00,-0.013495,0.000000,-0.013495,Loss
2,England v Ireland,England,Back then Lay,1.36,1.33,0.225564,0.011278,0.214286,Profit
3,England v Ireland,England,Lay then Back,1.36,1.33,-0.626566,0.000000,-0.626566,Loss
4,England v Ireland,Ireland,Back then Lay,4.20,4.40,-0.454545,0.000000,-0.454545,Loss
5,England v Ireland,Ireland,Lay then Back,4.20,4.40,0.142045,0.007102,0.134943,Profit
6,France v Ireland,Draw,Back then Lay,48.00,48.00,0.000000,0.000000,0.000000,Break-even
7,France v Ireland,Draw,Lay then Back,48.00,48.00,0.000000,0.000000,0.000000,Break-even
8,France v Ireland,France,Back then Lay,1.23,1.23,0.000000,0.000000,0.000000,Break-even
9,France v Ireland,France,Lay then Back,1.23,1.23,0.000000,0.000000,0.000000,Break-even


In [86]:
strategy_summary = trades_df.groupby("Strategy").agg(
    Number_of_Trades=("Net Green Profit", "count"),
    Profitable_Trades=(
        "Net Green Profit",
        lambda x: (x > 0).sum()
    ),
    Losing_Trades=(
        "Net Green Profit",
        lambda x: (x < 0).sum()
    ),
    Total_Gross_Profit=("Gross Green Profit", "sum"),
    Total_Commission=("Commission", "sum"),
    Total_Net_Profit=("Net Green Profit", "sum"),
    Average_Net_Profit=("Net Green Profit", "mean")
).reset_index()

strategy_summary["Win Rate %"] = (
    strategy_summary["Profitable_Trades"]
    / strategy_summary["Number_of_Trades"]
) * 100

strategy_summary

,Strategy,Number_of_Trades,Profitable_Trades,Losing_Trades,Total_Gross_Profit,Total_Commission,Total_Net_Profit,Average_Net_Profit,Win Rate %
0,Back then Lay,15,4,4,-1.978121,0.117497,-2.095619,-0.139708,26.666667
1,Lay then Back,15,4,4,1.906786,0.129941,1.776844,0.118456,26.666667


In [87]:
match_summary = trades_df.groupby(
    ["Match", "Strategy"]
).agg(
    Trades=("Net Green Profit", "count"),
    Net_Profit=("Net Green Profit", "sum"),
    Average_Profit=("Net Green Profit", "mean"),
    Profitable_Trades=(
        "Net Green Profit",
        lambda x: (x > 0).sum()
    )
).reset_index()

match_summary

,Match,Strategy,Trades,Net_Profit,Average_Profit,Profitable_Trades
0,England v Ireland,Back then Lay,3,0.259740,0.086580,2
1,England v Ireland,Lay then Back,3,-0.505119,-0.168373,1
2,France v Ireland,Back then Lay,3,0.161017,0.053672,1
3,France v Ireland,Lay then Back,3,-0.033898,-0.011299,0
4,Ireland v Italy,Back then Lay,3,1.357143,0.452381,1
5,Ireland v Italy,Lay then Back,3,-0.018083,-0.006028,0
6,Ireland v Scotland,Back then Lay,3,-1.470588,-0.490196,0
7,Ireland v Scotland,Lay then Back,3,0.049895,0.016632,1
8,Ireland v Wales,Back then Lay,3,-2.402930,-0.800977,0
9,Ireland v Wales,Lay then Back,3,2.284049,0.761350,2
